# Comparing the tests on legnini

Four ways to answer the same question on the same genes and the same expression layer: does a gene change differently with the distance to SHH+ cells in Ctrl and SHH slides?

| | Test | Fitted to | Inference | Slides | p-value |
|---|---|---|---|---|---|
| 1 | Wald | bins (sliding windows) | NUTS | ignored | Gamma null fitted to the panel |
| 2 | Curve test | bins (contiguous, weighted by cell count) | NUTS | ignored | per gene, noncentral chi-square |
| 3 | Curve test | cells | ADVI | ignored | per gene, noncentral chi-square |
| 4 | Curve test | cells | ADVI | empirical-Bayes random effects | per gene, noncentral F |

What the two tests look at:

* **Wald** (`test="wald"`) compares the two posterior-predictive condition curves. Its p-value comes from `T`, the mean squared gap between the curves, against a Gamma distribution fitted to the lower 90% of the panel. So `T` contains level differences as well as shape differences, and the p-value depends on which other genes are in the panel.
* **Curve test** (`test="parallel"`) is a Wald test on the interaction coefficients (`parallel_stat`), restricted to the directions that change the shape of the gap, against an interval null: `shape_diff <= tau`. Level differences do not enter, and the p-value is per gene, so no panel is needed.

Tests 2 and 3 are the same model on the same likelihood, once on weighted bins and once on cells, so they should agree. Tests 3 and 4 differ only in the slide effects.

Each test gets its ranked table, p-value histogram and volcano plot. The last section compares the four rankings.

**Runtime:** the slide model on cells is by far the slowest part, about 34 min with 4 workers on this data, against 3-7 min for each of the others. Run it on the cluster (`test_comparison_legnini.sbatch`), or shorten `GENES` for a trial, keeping 10 or more genes for the Gamma null.

In [ ]:
# One BLAS thread per process: the fits run in worker processes, and oversubscribed BLAS threads slow them down a lot.
# This has to happen before numpy is imported, so restart the kernel if numpy is already loaded.
import os

for var in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[var] = "1"

import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq

import diffsvg as dvg

## Data and anchor

Cells in the top percentile of SHH expression are the anchor. Every other cell gets its distance to the closest SHH+ cell of its own slide, scaled to 0-1 per slide. `slide1_A2-2` has no SHH+ cells, so it drops out of the fits.

In [ ]:
adata = sc.read_h5ad(os.environ.get("DIFFSVG_DATA", "/home/lehnerl/Arbeit/data/legnini23.h5ad"))

shh = adata[:, "SHH"].X
shh = (shh.toarray() if hasattr(shh, "toarray") else np.asarray(shh)).ravel()
adata.obs["SHH+"] = (shh > np.percentile(shh, 99)).astype(str)

sq.tl.var_by_distance(adata, cluster_key="SHH+", groups="True", library_key="sample", covariates="condition")
adata.obsm["design_matrix"].head()

## Settings

* `LAYER`: the Normal likelihood suits log-transformed data.
* `TAU`: the smallest shape difference that counts for the curve test, in slope-equivalent units. With `TAU > 0`, genes clearly below it get p-values near 1, so the histogram has a second peak there by design.
* `TOP_N`: the size of the top list in the comparison.

In [ ]:
LAYER = "log1p_norm"
GENES = list(adata.var_names)  # shorten for a trial run (keep 10+ genes for the Gamma null)
N_WORKERS = int(os.environ.get("DIFFSVG_WORKERS", 4))
TAU = 0.1
TOP_N = 20
RESULTS_PATH = os.environ.get("DIFFSVG_RESULTS", "test_comparison_legnini_results.pkl")

results, ranks, timings = {}, {}, {}

In [ ]:
PVALUE = {"wald": ("gamma_pval", "gamma_pval_adj", "T"), "parallel": ("parallel_pval", "parallel_pval_adj", "parallel_stat")}
COLUMNS = {
    "wald": ["T", "wald_stat", "gamma_pval", "gamma_pval_adj", "integral"],
    "parallel": ["shape_diff", "slope_diff", "parallel_stat", "parallel_pval", "parallel_pval_adj", "integral"],
}


def passes_diagnostics(stats):
    """Whether the fit of each gene can be trusted: R-hat for NUTS, the importance-sampling checks for ADVI."""
    if "vi_khat" not in stats:
        return stats["rhat_max"] < 1.05
    min_ess = 0.1 if "sample_level_sd" in stats else 0.5  # the ESS share falls with the number of parameters
    return (stats["vi_khat"] < 0.7) & (stats["vi_ess"] > min_ess)


def run_test(name, test, binning=None, **fit_kwargs):
    """Bin, fit and rank; every test keeps its own tables in `adata.varm`, so its curves can be plotted later."""
    key = f"test{len(results) + 1}"
    t0 = time.perf_counter()
    dvg.pp.bin_obs(adata, anchor="True", condition="condition", genes=GENES, layer=LAYER, n_bins=50, **(binning or {}))
    dvg.tl.fit_model(
        adata, dvg.tl.setup_model(adata), n_workers=N_WORKERS, test=test, tau=TAU,
        stats_key=f"{key}_stats", preds_key=f"{key}_preds", curves_key=f"{key}_curves", coefs_key=f"{key}_coefs",
        **fit_kwargs,
    )  # fmt: skip
    timings[name] = time.perf_counter() - t0

    stats = adata.varm[f"{key}_stats"].loc[GENES].copy()
    stats["passes_diagnostics"] = passes_diagnostics(stats)
    pvalue, _, statistic = PVALUE[test]
    # Ordered by p-value, with ties (p-values that underflow to 0, or sit at 1) broken by the larger statistic.
    table = stats.sort_values([pvalue, statistic], ascending=[True, False])
    table.insert(0, "rank", np.arange(1, len(table) + 1))
    results[name], ranks[name] = table, table["rank"]
    print(f"{timings[name]:.0f} s, {stats['passes_diagnostics'].sum()} of {len(stats)} genes pass the diagnostics")
    return table[["rank", *COLUMNS[test], "passes_diagnostics"]]


def qc_plots(name, test):
    """p-value histogram next to the volcano plot."""
    pvalue, pvalue_adj, _ = PVALUE[test]
    fig, axes = plt.subplots(1, 2, figsize=(11, 5))
    dvg.pl.plot_pvalue_histogram(results[name], pvalue=pvalue, ax=axes[0])
    dvg.pl.plot_volcano(results[name], pvalue=pvalue, pvalue_adj=pvalue_adj, ax=axes[1], n_label=15)
    fig.suptitle(name)
    fig.tight_layout()

## 1. Wald test on bins, NUTS

Sliding windows of radius 0.02 around 51 grid points (the `bin_obs` defaults), one model per gene, sampled with NUTS.

In [ ]:
NAME_1 = "1: Wald, bins, NUTS"
run_test(NAME_1, "wald").head(25)

In [ ]:
qc_plots(NAME_1, "wald")

## 2. Curve test on bins, NUTS

Contiguous bins (`window_radius=None`), so every cell counts once and each bin is weighted by its cell count. The curve test needs this: on the overlapping windows of test 1, neighbouring bins share cells and the p-values come out too small.

In [ ]:
NAME_2 = "2: Curve test, bins, NUTS"
run_test(NAME_2, "parallel", binning={"window_radius": None}).head(25)

In [ ]:
qc_plots(NAME_2, "parallel")

## 3. Curve test on cells, ADVI

The same test on every cell, with full-rank ADVI (Laplace start, Adam polish). `bin_obs` only provides the spline basis and the prediction grid here. Cells of a condition are pooled, so differences between slides count as differences between conditions, and with this many cells the test can only be held back by `tau`.

In [ ]:
NAME_3 = "3: Curve test, cells, ADVI"
run_test(NAME_3, "parallel", binning={"window_radius": None}, use_binned=False, inference="advi").head(25)

In [ ]:
qc_plots(NAME_3, "parallel")

## 4. Curve test on cells, ADVI with empirical-Bayes slide effects

`sample="sample"` adds a random level and a random curve per slide. Their two scales are set per gene by empirical Bayes (the mode of their marginal posterior) and then held fixed while ADVI fits the rest. The p-value uses an F reference with `n_slides - 2` denominator degrees of freedom, so the conditions are compared against the variation between slides rather than between cells. The plotted curves are the population-level ones.

In [ ]:
NAME_4 = "4: Curve test, cells, ADVI + EB"
table = run_test(
    NAME_4, "parallel", binning={"window_radius": None, "sample": "sample"},
    use_binned=False, inference="advi", slide_scales="empirical_bayes",
)  # fmt: skip
results[NAME_4][["rank", *COLUMNS["parallel"], "sample_level_sd", "sample_curve_sd", "parallel_df_denominator", "passes_diagnostics"]].head(25)

In [ ]:
qc_plots(NAME_4, "parallel")

The fits took long, so their tables are saved right away.

In [ ]:
pd.to_pickle({"results": results, "timings": timings}, RESULTS_PATH)
RESULTS_PATH

## Comparison

### Calls and runtime

How many genes each test flags, and what each one cost. The Wald p-values are not on the same footing as the curve-test ones: the Gamma null treats the lower 90% of the panel as null by construction, so the Wald test can only flag a small part of a panel.

In [ ]:
NAMES = list(results)
TEST_OF = {NAME_1: "wald", NAME_2: "parallel", NAME_3: "parallel", NAME_4: "parallel"}

pd.DataFrame(
    {
        name: {
            "p < 0.05": int((results[name][PVALUE[TEST_OF[name]][0]] < 0.05).sum()),
            "BH-adjusted < 0.05": int((results[name][PVALUE[TEST_OF[name]][1]] < 0.05).sum()),
            "pass the diagnostics": int(results[name]["passes_diagnostics"].sum()),
            "minutes": round(timings[name] / 60, 1),
        }
        for name in NAMES
    }
).T

### Rank correlation and overlap

Spearman correlation of the ranks over all genes, and the overlap of the top lists. The two say different things: most genes of the panel have no effect, so the lower part of each ranking is mostly noise and the correlation over all genes is partly a correlation of noise. The third matrix therefore repeats the correlation on the genes that are in the top list of at least one of the two tests.

In [ ]:
rank_table = pd.DataFrame(ranks).loc[GENES, NAMES]
top = {name: set(rank_table[name].nsmallest(TOP_N).index) for name in NAMES}

spearman = rank_table.corr(method="spearman")
overlap = pd.DataFrame({a: {b: len(top[a] & top[b]) for b in NAMES} for a in NAMES}).loc[NAMES, NAMES]
spearman_top = pd.DataFrame(
    {a: {b: rank_table.loc[sorted(top[a] | top[b]), [a, b]].corr(method="spearman").iloc[0, 1] for b in NAMES} for a in NAMES}
).loc[NAMES, NAMES]


def heatmap(ax, matrix, title, vmin, vmax, fmt):
    """Annotated matrix on one sequential scale; every cell carries its value."""
    ax.imshow(matrix.to_numpy(dtype=float), cmap="Blues", vmin=vmin, vmax=vmax)
    labels = [name.split(":")[0] for name in matrix.columns]
    ax.set_xticks(range(len(labels)), labels)
    ax.set_yticks(range(len(labels)), labels)
    ax.set_title(title, fontsize=11)
    ax.tick_params(length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)
    midpoint = (vmin + vmax) / 2
    for i, j in np.ndindex(matrix.shape):
        value = float(matrix.iloc[i, j])
        ax.text(j, i, format(value, fmt), ha="center", va="center", fontsize=10, color="white" if value > midpoint else "black")


fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
heatmap(axes[0], spearman, "Spearman, all genes", 0, 1, ".2f")
heatmap(axes[1], spearman_top, f"Spearman, union of the two top {TOP_N}", -1, 1, ".2f")
heatmap(axes[2], overlap, f"Shared genes in the top {TOP_N}", 0, TOP_N, ".0f")
fig.tight_layout()
print("\n".join(NAMES))

### Rank against rank

Every pair of tests, with the genes that move most labelled. Genes near the diagonal rank the same in both.

In [ ]:
PAIRS = [(a, b) for i, a in enumerate(NAMES) for b in NAMES[i + 1 :]]
N_MOVERS = 5

fig, axes = plt.subplots(2, 3, figsize=(13, 8.5))
for ax, (a, b) in zip(axes.ravel(), PAIRS, strict=True):
    x, y = rank_table[a], rank_table[b]
    ax.plot([1, len(x)], [1, len(x)], color="grey", lw=1)
    ax.scatter(x, y, s=18, color="steelblue", linewidth=0)
    for gene in (x - y).abs().nlargest(N_MOVERS).index:
        ax.annotate(gene, (x[gene], y[gene]), xytext=(3, 3), textcoords="offset points", fontsize=7)
    ax.set(xlabel=f"rank, {a}", ylabel=f"rank, {b}", title=f"Spearman {spearman.loc[a, b]:.2f}", aspect="equal")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
fig.tight_layout()

### Why Wald and the curve test disagree

`T` is the mean squared gap between the curves, so it splits into the squared level difference and the part from the shape: `T = integral^2 + shape_diff^2 / 12`. The first plot shows how much of `T` is level for every gene of the Wald fit. If the gap is mostly level, the Wald test can rank a gene highly although the curves run parallel, and the curve test will not flag it. The second plot checks whether this explains the disagreement: rank under the curve test minus rank under Wald, against the share of level in `T`.

In [ ]:
wald, curve = results[NAME_1], results[NAME_2].loc[results[NAME_1].index]
level_share = (wald["integral"] ** 2 / wald["T"]).clip(upper=1)
rank_shift = curve["rank"] - wald["rank"]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].hist(level_share, bins=20, color="steelblue", edgecolor="white")
axes[0].set(xlabel="share of T that is level (integral^2 / T)", ylabel="genes")
axes[1].axhline(0, color="grey", lw=1)
axes[1].scatter(level_share, rank_shift, s=18, color="steelblue", linewidth=0)
for gene in rank_shift.abs().nlargest(8).index:
    axes[1].annotate(gene, (level_share[gene], rank_shift[gene]), xytext=(3, 3), textcoords="offset points", fontsize=7)
axes[1].set(xlabel="share of T that is level", ylabel="rank under the curve test - rank under Wald")
for ax in axes:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
fig.tight_layout()
print(f"Spearman, level share vs. rank shift: {level_share.corr(rank_shift, method='spearman'):.2f}")

### What the slide effects change

The same fit with and without slide effects (tests 3 and 4). Slide effects leave the effect size roughly in place, because they mostly change how much the data say about it, so a gene whose difference comes from a few slides keeps its `shape_diff` and loses its significance. The table lists the genes flagged without slide effects that are no longer flagged with them.

In [ ]:
pooled, slides = results[NAME_3], results[NAME_4].loc[results[NAME_3].index]
floor = 1e-300

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
limit = 1.05 * float(max(pooled["shape_diff"].max(), slides["shape_diff"].max()))
axes[0].plot([0, limit], [0, limit], color="grey", lw=1)
axes[0].scatter(pooled["shape_diff"], slides["shape_diff"], s=18, color="steelblue", linewidth=0)
axes[0].set(xlabel="shape_diff, no slide effects", ylabel="shape_diff, EB slide effects")
axes[1].axhline(-np.log10(0.05), color="grey", lw=1)
axes[1].axvline(-np.log10(0.05), color="grey", lw=1)
axes[1].scatter(
    -np.log10(pooled["parallel_pval"].clip(lower=floor)),
    -np.log10(slides["parallel_pval"].clip(lower=floor)),
    s=18, color="steelblue", linewidth=0,
)  # fmt: skip
axes[1].set(xlabel="-log10 p, no slide effects", ylabel="-log10 p, EB slide effects")
for ax in axes:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
fig.tight_layout()

lost = (pooled["parallel_pval_adj"] < 0.05) & (slides["parallel_pval_adj"] >= 0.05)
pd.DataFrame(
    {
        "shape_diff, no slides": pooled.loc[lost, "shape_diff"],
        "shape_diff, EB": slides.loc[lost, "shape_diff"],
        "p, EB": slides.loc[lost, "parallel_pval"],
        "curve sd between slides": slides.loc[lost, "sample_curve_sd"],
    }
).sort_values("p, EB").round(3)

### The genes the tests disagree on most

The curves of the three genes with the largest spread in rank over the four tests, from the slide model. The bands hold the noise of a single cell or bin, so they are wider than the uncertainty of the curve itself: judge the difference by the test and the mean curves, not by whether the bands overlap.

In [ ]:
spread = rank_table.max(axis=1) - rank_table.min(axis=1)
disputed = spread.nlargest(3).index
display(rank_table.loc[disputed].assign(spread=spread[disputed]))
dvg.pl.plot_prediction_curve(adata, gene=list(disputed), curves_key="test4_curves", ncols=3);